In [18]:
from datasets import load_dataset

spanish_dataset = load_dataset("mteb/amazon_reviews_multi", "es")
english_dataset = load_dataset("mteb/amazon_reviews_multi", "en")
english_dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'label', 'label_text'],
        num_rows: 200000
    })
    validation: Dataset({
        features: ['id', 'text', 'label', 'label_text'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['id', 'text', 'label', 'label_text'],
        num_rows: 5000
    })
})

In [19]:
def show_samples(datasets, num_samples=3, seed=42):
    samples = datasets['train'].shuffle(seed=seed).select(range(num_samples))
    for sample in samples:
        print(f"{sample['text']}")

In [20]:
show_samples(english_dataset)

Worked in front position, not rear

3 stars because these are not rear brakes as stated in the item description. At least the mount adapter only worked on the front fork of the bike that I got it for.
meh

Does it’s job and it’s gorgeous but mine is falling apart, I had to basically put it together again with hot glue
Can't beat these for the money

Bought this for handling miscellaneous aircraft parts and hanger "stuff" that I needed to organize; it really fit the bill. The unit arrived quickly, was well packaged and arrived intact (always a good sign). There are five wall mounts-- three on the top and two on the bottom. I wanted to mount it on the wall, so all I had to do was to remove the top two layers of plastic drawers, as well as the bottom corner drawers, place it when I wanted and mark it; I then used some of the new plastic screw in wall anchors (the 50 pound variety) and it easily mounted to the wall. Some have remarked that they wanted dividers for the drawers, and that the

In [21]:
english_dataset.set_format('pandas')
english_df = english_dataset['train'][:]
english_dataset.reset_format()

english_df

,id,text,label,label_text
0,en_0964290,I'll spend twice the amount of time boxing up ...,0,0
1,en_0690095,Not use able\n\nthe cabinet dot were all detac...,0,0
2,en_0311558,The product is junk.\n\nI received my first or...,0,0
3,en_0044972,Fucking waste of money\n\nThis product is a pi...,0,0
4,en_0784379,bubble\n\nwent through 3 in one day doesn't fi...,0,0
...,...,...,...,...
199995,en_0046316,"Nice and fit as advertised\n\nCute slippers, m...",4,4
199996,en_0956024,good to keep the kids engaged\n\nMy 6 year old...,4,4
199997,en_0589358,This works\n\nReplaced my battery with it. Wor...,4,4
199998,en_0970602,"Well made.\n\nI like them, holding up well.",4,4


In [22]:
def split_title_body(example):
    title, _, body = example['text'].partition("\n\n")
    return {"review_title": title, "review_body": body, "stars": example["label"] + 1}

In [23]:
spanish_dataset = spanish_dataset.map(split_title_body, remove_columns=["text", "label", "label_text"])
english_dataset = english_dataset.map(split_title_body, remove_columns=["text", "label", "label_text"])
english_dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'review_title', 'review_body', 'stars'],
        num_rows: 200000
    })
    validation: Dataset({
        features: ['id', 'review_title', 'review_body', 'stars'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['id', 'review_title', 'review_body', 'stars'],
        num_rows: 5000
    })
})

In [24]:
from datasets import concatenate_datasets, DatasetDict

def sample(ds, n_train=10000, n_eval=500, seed=42):
    return DatasetDict({
        "train": ds["train"].shuffle(seed=seed).select(range(n_train)),
        "validation": ds["validation"].shuffle(seed=seed).select(range(n_eval)),
        "test": ds["test"].shuffle(seed=seed).select(range(n_eval)),
    })

english_reviews = sample(english_dataset)
spanish_reviews = sample(spanish_dataset)

In [25]:
reviews_dataset = DatasetDict()

for split in english_reviews.keys():
    reviews_dataset[split] = concatenate_datasets([english_reviews[split], spanish_reviews[split]])
    reviews_dataset[split] = reviews_dataset[split].shuffle(seed=42)

In [26]:
reviews_dataset = reviews_dataset.filter(
    lambda x: len(x["review_title"].split()) > 2 and len(x["review_body"].strip()) > 0
)
reviews_dataset

Filter: 100%|██████████| 1000/1000 [00:00<00:00, 40662.18 examples/s]


DatasetDict({
    train: Dataset({
        features: ['id', 'review_title', 'review_body', 'stars'],
        num_rows: 11447
    })
    validation: Dataset({
        features: ['id', 'review_title', 'review_body', 'stars'],
        num_rows: 565
    })
    test: Dataset({
        features: ['id', 'review_title', 'review_body', 'stars'],
        num_rows: 549
    })
})

In [27]:
from transformers import AutoTokenizer

model_checkpoint = "google/mt5-small"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

max_input_length = 512
max_target_length = 30

In [28]:
def preprocess_function(examples):
    model_inputs = tokenizer(examples["review_body"], max_length=max_input_length, truncation=True)
    labels = tokenizer(text_target=examples["review_title"], max_length=max_target_length, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [29]:
tokenized_datasets = reviews_dataset.map(
    preprocess_function, batched=True, remove_columns=reviews_dataset["train"].column_names
)
tokenized_datasets

Map: 100%|██████████| 549/549 [00:00<00:00, 8782.36 examples/s]


DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 11447
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 565
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 549
    })
})

In [30]:
%pip install rouge_score

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/1.8 MB ? eta -:--:--
   ----------- ---------------------------- 0.5/1.8 MB 953.4 kB/s eta 0:00:02
   ----------- ---------------------------- 0.5/1.8 MB 953.4 kB/s eta 0:00:02
   ----------------------- ---------------- 1.0/1.8 MB 1.1 MB/s eta 0:00:01
   ----------------------- ---------------- 1.0/1.8 MB 1.1 MB/s eta 0:00:01
   ----------------------- ---------------- 1.0/1.8 MB 1.1 MB/s eta 0:00:01
   ----------------------------- ---------- 1.3/1.8 MB 862.2 kB/s eta 0:00:01
 

In [31]:
import evaluate

metric = evaluate.load('rouge')

In [32]:
generated_summary = "I absolutely loved reading the Hunger Games"
reference_summary = "I loved reading the Hunger Games"

In [34]:
scores = metric.compute(
    predictions=[generated_summary], references=[reference_summary]
)
scores

{'rouge1': np.float64(0.923076923076923),
 'rouge2': np.float64(0.7272727272727272),
 'rougeL': np.float64(0.923076923076923),
 'rougeLsum': np.float64(0.923076923076923)}

In [35]:
%pip install nltk

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import nltk

nltk.download('')